Import and Config

In [0]:
from pyspark.sql import functions as F

catalog = "olist"
volume_path = f"/Volumes/{catalog}/bronze/raw_files"

files_to_tables = {
    "olist_orders_dataset.csv": "orders_raw",
    "olist_order_items_dataset.csv": "order_items_raw",
    "olist_order_payments_dataset.csv": "order_payments_raw",
    "olist_order_reviews_dataset.csv": "order_reviews_raw",
    "olist_customers_dataset.csv": "customers_raw",
    "olist_products_dataset.csv": "products_raw",
    "olist_sellers_dataset.csv": "sellers_raw",
    "olist_geolocation_dataset.csv": "geolocation_raw",
    "product_category_name_translation.csv": "category_translation_raw",
}

Loading

In [0]:
def load_to_bronze(file_name: str, table_name: str):
    df = (spark.read.format("csv")
          .option("header", True)
          .option("multiLine", True)
          .option("inferSchema", True)
          .load(f"{volume_path}/{file_name}"))

    df = (df.withColumn("_loaded_at", F.current_timestamp())
            .withColumn("_source", F.lit(file_name)))

    full_table_name = f"{catalog}.bronze.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)
    print(f"Loaded {file_name} -> {full_table_name} ({df.count()} rows)")

run it for every file

In [0]:
for file_name, table_name in files_to_tables.items():
    load_to_bronze(file_name, table_name)

sanity check

In [0]:
spark.sql("SHOW TABLES IN olist.bronze").display()